In [1]:
# preproc_tracetn.py
# Pre-Processing per il progetto tracetn (pendolarismo lavoro, Trentino)

from __future__ import annotations
import pandas as pd
import numpy as np
from pathlib import Path
import re

# === Percorsi di default (puoi modificarli se necessario) ===
DATA_DIR = Path("data")

FILE_MUNICIPALITIES = DATA_DIR / "municipalities_trentino.csv"
FILE_INCOMING      = DATA_DIR / "incoming_movements_2021.csv"
FILE_OUTCOMING     = DATA_DIR / "outcoming_movements_2021.csv"
FILE_POPULATION    = DATA_DIR / "trentino_population_2021.csv"
FILE_DIST_MATRIX   = DATA_DIR / "distance_km_matrix.csv"
FILE_TIME_MATRIX   = DATA_DIR / "time_hhmm_matrix.csv"


# -------------------------------
# Utilità
# -------------------------------
def _snakecase(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^\w]+", "_", s, flags=re.UNICODE)
    s = re.sub(r"_+", "_", s)
    return s.strip("_").lower()

def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_snakecase(c) for c in df.columns]
    return df

def _guess_id_col(df: pd.DataFrame) -> str:
    candidates = ["id", "istat", "istat_code", "cod_istat", "code", "id_comune", "id_istat"]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: se esiste una colonna 'comune' + indexing
    if "comune" in df.columns:
        # creeremo un id fittizio solo per procedere (sconsigliato in produzione)
        return None
    raise ValueError("Colonna ID del comune non trovata. Aggiungi una colonna id/istat_code/cod_istat ecc.")

def _ensure_int(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

def _validate_lordo(df: pd.DataFrame, col_stesso: str, col_altro: str, col_totale: str, col_lordo: str,
                    dataset_name: str, tol: int = 0):
    if not set([col_stesso, col_altro, col_totale, col_lordo]).issubset(df.columns):
        return
    calc = df[col_stesso] + df[col_altro] + df[col_totale]
    diff = (df[col_lordo] - calc).abs()
    n_bad = (diff > tol).sum()
    if n_bad > 0:
        bad_rows = df.loc[diff > tol].index.tolist()
        raise ValueError(
            f"[{dataset_name}] Incoerenza Lordo != Stesso+Altro+Totale in {n_bad} righe. Indici problematici: {bad_rows[:10]} ..."
        )

def _hhmm_to_hours(x: str | float | int) -> float:
    """
    Converte stringhe 'H:MM' o 'HH:MM' in ore (float). Se già numerico, ritorna float.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if ":" not in s:
        # prova anche 'H.MM' etc.
        s = s.replace(",", ".")
        try:
            return float(s)
        except:
            return np.nan
    try:
        h, m = s.split(":")
        return int(h) + int(m)/60.0
    except:
        return np.nan

def _align_square_matrix(mat: pd.DataFrame, ids: list, label: str) -> pd.DataFrame:
    """
    Allinea righe/colonne della matrice all'ordine 'ids'.
    - Se le intestazioni sono stringhe, prova a convertirle a numeri quando opportuno.
    - Se mancano id, crea colonne/righe con NaN.
    """
    mat = mat.copy()

    # Prova a uniformare header a stringhe pulite e poi a confrontare
    mat.columns = [str(c).strip() for c in mat.columns]
    mat.index   = [str(i).strip() for i in mat.index]

    ids_str = [str(i) for i in ids]

    # Se gli header sono nomi di comuni anziché id, si può tentare un mapping esterno in fase successiva.
    # Qui assumiamo che siano ID coerenti.
    missing_rows = [i for i in ids_str if i not in mat.index]
    missing_cols = [i for i in ids_str if i not in mat.columns]

    if missing_rows:
        # Aggiungi righe mancanti piene di NaN
        for r in missing_rows:
            mat.loc[r] = np.nan

    if missing_cols:
        # Aggiungi colonne mancanti piene di NaN
        for c in missing_cols:
            mat[c] = np.nan

    # Riordina
    mat = mat.loc[ids_str, ids_str]

    # Verifica quadratura
    if mat.shape[0] != mat.shape[1]:
        raise ValueError(f"[{label}] La matrice non è quadrata dopo l'allineamento: {mat.shape}")

    return mat


# -------------------------------
# Pre-processing principale
# -------------------------------
def preprocess(
    file_municipalities: Path = FILE_MUNICIPALITIES,
    file_incoming: Path = FILE_INCOMING,
    file_outcoming: Path = FILE_OUTCOMING,
    file_population: Path = FILE_POPULATION,
    file_dist_matrix: Path = FILE_DIST_MATRIX,
    file_time_matrix: Path = FILE_TIME_MATRIX
) -> dict:
    """
    Esegue il pre-processing dei 6 file:
    - uniforma colonne
    - valida coerenza Lordo
    - unisce incoming/outcoming/popolazione a municipalities
    - carica e allinea le matrici distanza/tempo
    - converte tempi H:MM in ore (float)

    Ritorna un dict con:
      municipalities, incoming, outcoming, population,
      df_communes (tabella integrata per comune),
      dist_km (matrice allineata),
      time_h (matrice allineata, ore float)
    """

    # --- 1) Caricamento e standardizzazione colonne
    municipalities = _standardize_columns(pd.read_csv(file_municipalities))
    incoming       = _standardize_columns(pd.read_csv(file_incoming))
    outcoming      = _standardize_columns(pd.read_csv(file_outcoming))
    population     = _standardize_columns(pd.read_csv(file_population))

    # Matrici: manteniamo raw poi trasformiamo
    dist_km_raw    = pd.read_csv(file_dist_matrix, index_col=0)
    time_raw       = pd.read_csv(file_time_matrix, index_col=0)

    # --- 2) Identificazione colonne id e nome
    id_col_muni = _guess_id_col(municipalities)
    if id_col_muni is None:
        # crea id sintetico SE manca; meglio sostituire con codice ISTAT reale
        municipalities = municipalities.reset_index().rename(columns={"index": "id"})
        id_col_muni = "id"

    # Prova a normalizzare un possibile 'comune' come name
    name_col = "name"
    if name_col not in municipalities.columns:
        for c in ["comune", "municipality", "nome", "denominazione"]:
            if c in municipalities.columns:
                municipalities = municipalities.rename(columns={c: name_col})
                break

    # Uniforma tipi id (stringa per evitare mismatch tra int/str)
    municipalities[id_col_muni] = municipalities[id_col_muni].astype(str)

    # --- 3) Incoming/Outcoming: rinomina colonne chiave secondo convenzione
    # ci aspettiamo: 'stesso', 'altro', 'totale', 'lordo', 'netto' + id
    # prova a trovare la colonna id:
    def _standardize_io(df: pd.DataFrame, label: str) -> pd.DataFrame:
        df = df.copy()
        id_col = _guess_id_col(df)
        if id_col is None:
            # se manca, e c'è 'comune', associa via nome dopo
            pass
        # mappa possibili sinonimi
        rename_map = {}
        for col in df.columns:
            c = col
            if c in {"stesso", "same"}:
                rename_map[col] = "stesso"
            elif c in {"altro", "inner", "trentino", "altri_comuni"}:
                rename_map[col] = "altro"
            elif c in {"totale", "fuori", "extra", "fuori_provincia"}:
                rename_map[col] = "totale"
            elif c in {"lordo", "total", "sum"}:
                rename_map[col] = "lordo"
            elif c in {"netto", "pendolari", "non_residenti"}:
                rename_map[col] = "netto"
        df = df.rename(columns=rename_map)

        # vincola tipi interi dove presenti
        for c in ["stesso", "altro", "totale", "lordo", "netto"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

        # id come stringa
        if id_col is not None:
            df[id_col] = df[id_col].astype(str)

        return df

    incoming = _standardize_io(incoming, "incoming")
    outcoming = _standardize_io(outcoming, "outcoming")

    # --- 4) Validazioni base sui dataset incoming/outcoming
    _validate_lordo(incoming, "stesso", "altro", "totale", "lordo", "incoming")
    _validate_lordo(outcoming, "stesso", "altro", "totale", "lordo", "outcoming")

    # --- 5) Popolazione: trova colonna popolazione
    pop_col_candidates = ["population", "popolazione", "residenti", "pop_res"]
    pop_col = None
    for c in pop_col_candidates:
        if c in population.columns:
            pop_col = c
            break
    if pop_col is None:
        raise ValueError("Colonna della popolazione non trovata in trentino_population_2021.csv")

    pop_id_col = _guess_id_col(population)
    if pop_id_col is None:
        raise ValueError("Colonna ID non trovata in trentino_population_2021.csv")
    population[pop_id_col] = population[pop_id_col].astype(str)

    # --- 6) Merge tabella per-comune (df_communes)
    # unione left su municipalities per mantenere l'ordine/insieme dei comuni
    df_communes = municipalities[[id_col_muni] + ([name_col] if name_col in municipalities.columns else [])].copy()

    # prova a unire su id; se incoming/outcoming non hanno id ma 'comune', occorre un join per nome
    def _safe_merge(left: pd.DataFrame, right: pd.DataFrame, right_label: str) -> pd.DataFrame:
        left = left.copy()
        right = right.copy()

        left_id  = id_col_muni
        right_id = _guess_id_col(right)

        if right_id is not None:
            right[right_id] = right[right_id].astype(str)
            merged = left.merge(right, how="left", left_on=left_id, right_on=right_id, suffixes=("", f"_{right_label}"))
            # se right_id è diverso da left_id, rimuovi la colonna duplicata
            if right_id != left_id:
                merged = merged.drop(columns=[right_id])
        else:
            # fallback via nome comune, se esiste in entrambi
            if (name_col in left.columns) and ("comune" in right.columns):
                merged = left.merge(right.rename(columns={"comune": name_col}), how="left", on=name_col,
                                    suffixes=("", f"_{right_label}"))
            else:
                raise ValueError(f"Impossibile unire {right_label}: manca id e non c'è 'comune' per join su nome.")
        return merged

    df_communes = _safe_merge(df_communes, incoming, "in")
    df_communes = _safe_merge(df_communes, outcoming, "out")
    df_communes = _safe_merge(df_communes, population[[pop_id_col, pop_col]], "pop")

    # rinomina colonne chiave per chiarezza
    rename_pairs = {
        "stesso": "in_stesso",
        "altro": "in_altro",
        "totale": "in_totale",
        "lordo": "in_lordo",
        "netto": "in_netto",
        "stesso_out": "out_stesso",
        "altro_out": "out_altro",
        "totale_out": "out_totale",
        "lordo_out": "out_lordo",
        "netto_out": "out_netto",
        pop_col: "population"
    }

    # Applica rinominhe robuste (considera suffissi _in / _out derivati dal merge)
    df_communes = df_communes.rename(columns={
        "stesso": "in_stesso", "altro": "in_altro", "totale": "in_totale",
        "lordo": "in_lordo", "netto": "in_netto",
        "stesso_out": "out_stesso", "altro_out": "out_altro", "totale_out": "out_totale",
        "lordo_out": "out_lordo", "netto_out": "out_netto",
        pop_col: "population"
    })

    # Se per qualche motivo alcune colonne _out sono ancora senza suffisso, prova a mapparle
    for base in ["stesso", "altro", "totale", "lordo", "netto"]:
        c_out = f"{base}_out"
        if c_out in df_communes.columns and f"out_{base}" not in df_communes.columns:
            df_communes = df_communes.rename(columns={c_out: f"out_{base}"})
    # idem per _in
    for base in ["stesso", "altro", "totale", "lordo", "netto"]:
        c_in = f"{base}_in"
        if c_in in df_communes.columns and f"in_{base}" not in df_communes.columns:
            df_communes = df_communes.rename(columns={c_in: f"in_{base}"})

    # --- 7) Tipi numerici coerenti su colonne attese
    int_cols = [
        "in_stesso","in_altro","in_totale","in_lordo","in_netto",
        "out_stesso","out_altro","out_totale","out_lordo","out_netto",
        "population"
    ]
    for c in int_cols:
        if c in df_communes.columns:
            df_communes[c] = pd.to_numeric(df_communes[c], errors="coerce").fillna(0).astype(int)

    # --- 8) Allineamento matrici distanza/tempo all'ordine dei comuni
    # Ordine degli id come stringhe
    ids_order = df_communes[id_col_muni].astype(str).tolist()

    # Distanze (km) - devono essere numeriche
    dist_km = dist_km_raw.apply(pd.to_numeric, errors="coerce")
    dist_km = _align_square_matrix(dist_km, ids_order, "distance_km_matrix")

    # Tempi: converti H:MM -> ore float
    time_h = time_raw.copy()
    time_h = time_h.applymap(_hhmm_to_hours)
    time_h = _align_square_matrix(time_h, ids_order, "time_hhmm_matrix")

    # --- 9) Riepilogo base (opzionale da loggare)
    # print("Comuni:", len(df_communes))
    # print("Colonne df_communes:", df_communes.columns.tolist())
    # print("Shape dist_km:", dist_km.shape, " | time_h:", time_h.shape)

    return {
        "municipalities": municipalities,
        "incoming": incoming,
        "outcoming": outcoming,
        "population": population,
        "df_communes": df_communes,   # tabella integrata per-comune, pronta per calcolo indicatori
        "dist_km": dist_km,           # matrice km allineata all'ordine dei comuni
        "time_h": time_h,             # matrice ore (float) allineata all'ordine dei comuni
        "id_col": id_col_muni,
        "name_col": name_col if name_col in municipalities.columns else None
    }


# Esempio d'uso (da notebook o script):
# data = preprocess()
# df = data["df_communes"]
# df.head()
